# Dataset Overview
This notebook scans `../data`, computes features using `python/features.py`, and visualizes distributions.

In [ ]:
# Imports and setup
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on the path so `python.features` can be imported
sys.path.append(str(Path('..').resolve()))

plt.style.use('seaborn-v0_8')
from resonancedb.features import compute_features


In [ ]:
# Find JSON files under ../data
data_dir = Path('../data').resolve()
json_files = list(data_dir.rglob('*.json'))
print(f'Found {len(json_files)} JSON files in {data_dir}')
for f in json_files[:10]:
    print('-', f.name)


In [ ]:
# Load and compute features
rows = []
for fp in json_files:
    try:
        with open(fp, 'r') as f:
            d = json.load(f)
    except Exception as e:
        print(f'Failed to load {fp}: {e}')
        continue
    material = d.get('material', 'unknown')
    vib = np.array(d.get('vibration', []), dtype=float)
    sr = d.get('sample_rate_hz', None)
    n = len(vib)
    if sr and n > 0:
        feats = compute_features(vib, sr)
    else:
        feats = {'peak_freq': np.nan, 'decay_rate': np.nan, 'energy': np.nan}
    rows.append({
        'file': fp.name,
        'material': material,
        'sample_rate_hz': sr,
        'n_samples': n,
        **feats
    })
df = pd.DataFrame(rows)
df.head()


In [ ]:
# Summary stats
print('Materials:', sorted(df['material'].unique()))
print('Counts per material:')
print(df['material'].value_counts())
print('Sample rates:')
print(df['sample_rate_hz'].value_counts())
df.describe(include='all')


In [ ]:
# Histograms of core features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['peak_freq'].plot(kind='hist', bins=20, ax=axes[0], color='steelblue')
axes[0].set_title('Peak Frequency (Hz)')
df['decay_rate'].plot(kind='hist', bins=20, ax=axes[1], color='salmon')
axes[1].set_title('Decay Rate')
df['energy'].plot(kind='hist', bins=20, ax=axes[2], color='seagreen')
axes[2].set_title('Energy')
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots by material
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for m, sub in df.groupby('material'):
    axes[0].scatter(sub['peak_freq'], sub['energy'], label=m, alpha=0.8)
axes[0].set_xlabel('Peak Freq (Hz)')
axes[0].set_ylabel('Energy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for m, sub in df.groupby('material'):
    axes[1].scatter(sub['decay_rate'], sub['energy'], label=m, alpha=0.8)
axes[1].set_xlabel('Decay Rate')
axes[1].set_ylabel('Energy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
